<a href="https://colab.research.google.com/github/shankar-011/GenAI-Internship/blob/master/Day7/SentimentAnalysisDL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [2]:
vocab_size=10000
#Keep only the top 10000 most frequent words
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)
print(' training sequences:', len(X_train))
print('test sequences:', len(X_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
 training sequences: 25000
test sequences: 25000


In [3]:
max_length=200
X_train=pad_sequences(X_train, maxlen=max_length,padding='post')
X_test=pad_sequences(X_test, maxlen=max_length,padding='post')

In [4]:
model=Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length),
    SimpleRNN(64),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [5]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [6]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:
history=model.fit(X_train,y_train,epochs=5,validation_split=0.2,batch_size=64)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 27s 78ms/step - accuracy: 0.5010 - loss: 0.6960 - val_accuracy: 0.5062 - val_loss: 0.6952
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 41s 78ms/step - accuracy: 0.5150 - loss: 0.6903 - val_accuracy: 0.5356 - val_loss: 0.6855
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 24s 77ms/step - accuracy: 0.5640 - loss: 0.6702 - val_accuracy: 0.5506 - val_loss: 0.6973
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 24s 75ms/step - accuracy: 0.6169 - loss: 0.6388 - val_accuracy: 0.5870 - val_loss: 0.6528
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 42s 77ms/step - accuracy: 0.6439 - loss: 0.5778 - val_accuracy: 0.5986 - val_loss: 0.6381


In [8]:
loss,accuracy=model.evaluate(X_test,y_test)
print('Test accuracy:', accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.5989 - loss: 0.6367
Test accuracy: 0.5989199876785278


In [9]:
from tensorflow.keras.datasets import imdb
word_index=imdb.get_word_index()

#shift indices by 3 because keras reserves 0,1,2
word_index={k:(v+3) for k,v in word_index.items()}
word_index['<PAD>']=0
word_index['<START>']=1
word_index['<UNK>']=2
word_index['<UNUSED>']=3

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [10]:
def review_to_sequence(review):
  review=review.lower().split()
  sequence=[]
  for word in review:
    sequence.append(word_index.get(word,2)) #2=unknown word
  return sequence

In [11]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
max_length=200
def predict_sentiment(review):
  sequence=review_to_sequence(review)
  padded=pad_sequences(
      [sequence],
      maxlen=max_length,
      padding='post',
      truncating='post'
  )
  prediction=model.predict(padded,verbose=0)
  score=prediction[0][0]
  print("Sentiment Score",score)
  if score>0.5:
    print("Positive review")
  else:
    print("Negative review")

In [12]:
predict_sentiment("This movie is boring")

Sentiment Score 0.46060416
Negative review
